# Benchmark Comparativo de Índices FAISS — Vibe Bridge

Este notebook avalia e compara três arquiteturas do **FAISS** para a busca de vizinhos mais próximos no espaço vetorial do Spotify:
1. **IndexFlatL2:** Distância Euclidiana Exata (Baseline)
2. **IndexFlatIP:** Produto Interno / Similaridade por Cosseno
3. **IndexIVFFlat:** Busca Aproximada por Agrupamento (Inverted File Index)

In [1]:
import pandas as pd
import numpy as np
import faiss
import time
from sklearn.preprocessing import StandardScaler

# 1. Carregar dataset pré-processado
df_clean = pd.read_parquet("spotify_tracks_clean.parquet")

FEATURE_COLS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# 2. Padronização Z-score
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURE_COLS]).astype('float32')
dimension = X_scaled.shape[1]

# 3. Definir Vetor Alvo para o Benchmark (Ponto médio alpha = 0.5 entre faixa 0 e 100)
v_A = X_scaled[[0]]
v_B = X_scaled[[100]]
vetor_target = (0.5 * v_A + 0.5 * v_B).astype('float32')

print(f"Dataset pronto com {len(df_clean)} faixas e {dimension} dimensões vetoriais.")

Dataset pronto com 89741 faixas e 9 dimensões vetoriais.


## 1. Abordagem 1 — IndexFlatL2 (Distância Euclidiana Exata)

Calcula a distância geométrica absoluta em linha reta no espaço de 9 dimensões. É o modelo padrão (*baseline*) do Vibe Bridge.

In [2]:
# Criar e popular o índice L2
index_l2 = faiss.IndexFlatL2(dimension)
index_l2.add(X_scaled)

# Medir tempo de consulta
inicio = time.time()
dist_l2, idx_l2 = index_l2.search(vetor_target, k=5)
tempo_l2 = (time.time() - inicio) * 1000

print(f"⏱️ Tempo de consulta (L2): {tempo_l2:.3f} ms\n")
df_clean.iloc[idx_l2[0]][['track_name', 'artists', 'popularity', 'energy', 'valence', 'tempo']]

⏱️ Tempo de consulta (L2): 1.640 ms



,track_name,artists,popularity,energy,valence,tempo
33812,menina solta,GIULIA BE,0,0.576,0.668,91.953
14664,Shortcut To Heaven,lullaboy,63,0.505,0.565,91.990
49294,Jake from State Farm,salem ilese,26,0.623,0.637,87.960
76778,Eu Nunca Amei Assim,RDN;Suel;Ferrugem,37,0.543,0.651,85.923
19193,Lonely,Akon,81,0.526,0.623,90.087


## 2. Abordagem 2 — IndexFlatIP (Similaridade por Cosseno / Produto Interno)

Mede a orientação do vetor em vez da magnitude geométrica. Para corresponder à similaridade por cosseno, os vetores precisam ser normalizados com norma unitária ($L_2 = 1$).

In [3]:
# Normalizar matriz de características e vetor alvo para norma unitária L2
X_norm = X_scaled.copy()
faiss.normalize_L2(X_norm)

v_target_norm = vetor_target.copy()
faiss.normalize_L2(v_target_norm)

# Criar e popular o índice IP (Inner Product)
index_ip = faiss.IndexFlatIP(dimension)
index_ip.add(X_norm)

# Medir tempo de consulta
inicio = time.time()
dist_ip, idx_ip = index_ip.search(v_target_norm, k=5)
tempo_ip = (time.time() - inicio) * 1000

print(f"⏱️ Tempo de consulta (IP): {tempo_ip:.3f} ms\n")
df_clean.iloc[idx_ip[0]][['track_name', 'artists', 'popularity', 'energy', 'valence', 'tempo']]

⏱️ Tempo de consulta (IP): 0.305 ms



,track_name,artists,popularity,energy,valence,tempo
78790,Numb Little Bug,Em Beihold,66,0.527,0.638,84.974
78730,Numb Little Bug,Em Beihold,0,0.527,0.638,84.974
78721,Numb Little Bug,Em Beihold,2,0.527,0.638,84.974
78668,Numb Little Bug,Em Beihold,78,0.527,0.638,84.974
33812,menina solta,GIULIA BE,0,0.576,0.668,91.953


## 3. Abordagem 3 — IndexIVFFlat (Inverted File Index / Busca Aproximada)

Divide o espaço multidimensional em $n_{list}$ clusters utilizando $K$-means. No momento da busca, verifica apenas os $n_{probe}$ clusters mais próximos, garantindo altíssima escalabilidade para grandes catálogos.

In [4]:
nlist = 50  # Quantidade de clusters para particionar a base
quantizer = faiss.IndexFlatL2(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_L2)

# Treinar os centroides com os vetores do dataset
index_ivf.train(X_scaled)
index_ivf.add(X_scaled)

# Configurar o número de clusters a serem consultados por busca
index_ivf.nprobe = 5

# Medir tempo de consulta
inicio = time.time()
dist_ivf, idx_ivf = index_ivf.search(vetor_target, k=5)
tempo_ivf = (time.time() - inicio) * 1000

print(f"⏱️ Tempo de consulta (IVFFlat): {tempo_ivf:.3f} ms\n")
df_clean.iloc[idx_ivf[0]][['track_name', 'artists', 'popularity', 'energy', 'valence', 'tempo']]

⏱️ Tempo de consulta (IVFFlat): 1.177 ms



,track_name,artists,popularity,energy,valence,tempo
33812,menina solta,GIULIA BE,0,0.576,0.668,91.953
14664,Shortcut To Heaven,lullaboy,63,0.505,0.565,91.990
76778,Eu Nunca Amei Assim,RDN;Suel;Ferrugem,37,0.543,0.651,85.923
19193,Lonely,Akon,81,0.526,0.623,90.087
78668,Numb Little Bug,Em Beihold,78,0.527,0.638,84.974


## 4. Tabela Comparativa de Desempenho e Recomendação

In [5]:
df_benchmark = pd.DataFrame({
    'Abordagem FAISS': [
        'IndexFlatL2 (Euclidiana Exata)',
        'IndexFlatIP (Cosseno / Prod. Interno)',
        'IndexIVFFlat (Clusters / ANN)'
    ],
    'Tempo de Consulta (ms)': [
        f"{tempo_l2:.4f}",
        f"{tempo_ip:.4f}",
        f"{tempo_ivf:.4f}"
    ],
    'Primeira Escolha Retornada': [
        f"{df_clean.iloc[idx_l2[0][0]]['track_name']} - {df_clean.iloc[idx_l2[0][0]]['artists']}",
        f"{df_clean.iloc[idx_ip[0][0]]['track_name']} - {df_clean.iloc[idx_ip[0][0]]['artists']}",
        f"{df_clean.iloc[idx_ivf[0][0]]['track_name']} - {df_clean.iloc[idx_ivf[0][0]]['artists']}"
    ]
})

df_benchmark

,Abordagem FAISS,Tempo de Consulta (ms),Primeira Escolha Retornada
0,IndexFlatL2 (Euclidiana Exata),1.6403,menina solta - GIULIA BE
1,IndexFlatIP (Cosseno / Prod. Interno),0.3054,Numb Little Bug - Em Beihold
2,IndexIVFFlat (Clusters / ANN),1.1771,menina solta - GIULIA BE
